# BB test on tick data

In [35]:
from pathlib import Path
import pandas as pd
import datetime

In [36]:
date_ = "17APR2026"
file_name = "NIFTY 50.xlsx"
file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\extracted_symbols\{file_name}")
nifty_all = pd.read_excel(file_path)

In [37]:
nifty_all.columns

Index(['instrument_token', 'symbol', 'exchange_timestamp', 'local_time',
       'last_trade_time', 'last_price', 'option_CE_PE', 'option_type',
       'strike', 'ohlc', 'change', 'tradable', 'mode'],
      dtype='str')

In [38]:
nifty_all.head()

,instrument_token,symbol,exchange_timestamp,local_time,last_trade_time,last_price,option_CE_PE,option_type,strike,ohlc,change,tradable,mode
0,256265,NIFTY 50,NaN,2026-04-17 09:10:05.827,NaN,24165.9,NaN,index,NaN,"{'high': 24165.9, 'low': 24165.9, 'open': 2416...",-0.127496,False,full
1,256265,NIFTY 50,NaN,2026-04-17 09:10:06.078,NaN,24165.9,NaN,index,NaN,"{'high': 24165.9, 'low': 24165.9, 'open': 2416...",-0.127496,False,full
2,256265,NIFTY 50,NaN,2026-04-17 09:10:06.596,NaN,24165.9,NaN,index,NaN,"{'high': 24165.9, 'low': 24165.9, 'open': 2416...",-0.127496,False,full
3,256265,NIFTY 50,NaN,2026-04-17 09:10:07.077,NaN,24165.9,NaN,index,NaN,"{'high': 24165.9, 'low': 24165.9, 'open': 2416...",-0.127496,False,full
4,256265,NIFTY 50,NaN,2026-04-17 09:10:07.577,NaN,24165.9,NaN,index,NaN,"{'high': 24165.9, 'low': 24165.9, 'open': 2416...",-0.127496,False,full


In [39]:
type(nifty_all.iloc[0]["local_time"])

pandas.Timestamp

In [40]:
nifty_all.iloc[4]["local_time"].hour, nifty_all.iloc[4]["local_time"].minute, nifty_all.iloc[4]["local_time"].second

(9, 10, 7)

In [41]:
# nifty_all["local_time"] = pd.to_datetime(nifty_all["local_time"])
# nifty_all["hhmmss"] = nifty_all["local_time"].dt.strftime("%H%M%S")
# nifty_all["hhmm"] = nifty_all["local_time"].dt.strftime("%H%M")

In [42]:
# nifty_all.to_excel("nifty_all_dev.xlsx")

In [43]:
import pandas as pd
import numpy as np
import pandas_ta as ta

# =========================================================
# HELPERS
# =========================================================

def tradingview_roc(close, length=10):
    return 100 * (close / close.shift(length) - 1)

def tradingview_bb(close, length=20, mult=2.0):
    basis = close.rolling(length).mean()
    std = close.rolling(length).std(ddof=0)
    dev = mult * std
    upper = basis + dev
    lower = basis - dev
    return basis, upper, lower
# =========================================================
# MAIN ENGINE: TICK-EVOLVING INDICATORS
# =========================================================

def generate_tick_evolving_signals(tick_df, dmi_length=14, roc_length=10, bb_length=20, bb_mult=2.0):

    tick_df = tick_df.copy()
    tick_df["local_time"] = pd.to_datetime(tick_df["local_time"])
    tick_df = tick_df.sort_values("local_time").reset_index(drop=True)

    # import datetime

    start = datetime.time(9, 15)
    end   = datetime.time(15, 31)

    tick_df = tick_df[
        (tick_df["local_time"].dt.time >= start) &
        (tick_df["local_time"].dt.time <= end)
    ]


    # Storage
    results = []

    # Rolling candle storage
    candles = []

    current_minute = None
    current_candle = None

    for i, row in tick_df.iterrows():

        ts = row["local_time"]
        price = row["last_price"]
        minute = ts.floor("min")

        # -------------------------------------------------
        # NEW MINUTE → finalize previous candle
        # -------------------------------------------------
        if current_minute is None or minute != current_minute:

            if current_candle is not None:
                candles.append(current_candle)

            # start new candle
            current_candle = {
                "date": minute,
                "open": price,
                "high": price,
                "low": price,
                "close": price
            }

            current_minute = minute

        else:
            # update current candle
            current_candle["high"] = max(current_candle["high"], price)
            current_candle["low"] = min(current_candle["low"], price)
            current_candle["close"] = price

        # -------------------------------------------------
        # BUILD TEMP DF INCLUDING CURRENT PARTIAL CANDLE
        # -------------------------------------------------
        temp_candles = candles + [current_candle]
        temp_df = pd.DataFrame(temp_candles)

        # Need enough history
        if len(temp_df) < max(dmi_length, roc_length) + 5:
            results.append(None)
            continue

        # -------------------------------------------------
        # COMPUTE INDICATORS (INCLUDING CURRENT PARTIAL)
        # -------------------------------------------------
        dmi = ta.adx(temp_df["high"], temp_df["low"], temp_df["close"], length=dmi_length)

        if dmi is None or dmi.empty:
            results.append(None)
            continue

        temp_df["plus_di"]  = dmi[[c for c in dmi.columns if "DMP" in c][0]]
        temp_df["minus_di"] = dmi[[c for c in dmi.columns if "DMN" in c][0]]
        temp_df["adx"]      = dmi[[c for c in dmi.columns if "ADX" in c][0]]
        temp_df["roc"] = tradingview_roc(temp_df["close"], roc_length)
        temp_df['basis'], temp_df['upper_bb'], temp_df['lower_bb'] = tradingview_bb(
            temp_df['close'], bb_length, bb_mult
        )
        # -------------------------------------------------
        # GET CURRENT + PREVIOUS VALUES
        # -------------------------------------------------
        curr = temp_df.iloc[-1]
        prev = temp_df.iloc[-2]

        # -------------------------------------------------
        # SIGNAL LOGIC (NOW TRULY EVOLVING) - ADX and DI
        # -------------------------------------------------
        # long_signal = (
        #     (curr["minus_di"] < 0.95 * prev["minus_di"]) and
        #     (curr["roc"] > prev["roc"] + 0.01)
        # )
        #
        # short_signal = (
        #     (curr["plus_di"] < 0.95 * prev["plus_di"]) and
        #     (curr["roc"] < prev["roc"] - 0.01)
        # )
        # -------------------------------------------------
        # SIGNAL LOGIC (NOW TRULY EVOLVING) - only BB ADX DI
        # -------------------------------------------------
        long_signal = (
            (curr["close"] < prev["lower_bb"]) and
            (curr["open"] < curr["lower_bb"]) and
            (curr["minus_di"] < 0.95 * prev["minus_di"]) and
            (curr["roc"] > prev["roc"] + 0.01)
        )

        short_signal = (
            (prev["close"] > prev["upper_bb"]) and
            (curr["open"] > curr["upper_bb"]) and
            (curr["plus_di"] < 0.95 * prev["plus_di"]) and
            (curr["roc"] < prev["roc"] - 0.01)
        )
        if long_signal:
            results.append("BUY")
        elif short_signal:
            results.append("SELL")
        else:
            results.append(None)

    tick_df["signal"] = results
    tick_df["hour"] = tick_df["local_time"].dt.hour
    tick_df["minute"] = tick_df["local_time"].dt.minute
    tick_df["second"] = tick_df["local_time"].dt.second

    return tick_df


In [44]:
nifty_all = generate_tick_evolving_signals(nifty_all)

In [45]:
from pathlib import Path
import os

path_obj = Path(file_name)
new_file_name = path_obj.with_stem(f"{path_obj.stem}_signal").name
file_path = Path(fr"D:\Study\Programs\trading\assets\logs\{date_}\signals\BB_ADX_DI_{new_file_name}")
os.makedirs(os.path.dirname(file_path), exist_ok=True)
nifty_all.to_excel(file_path)

In [46]:
new_file_name

'NIFTY 50_signal.xlsx'

In [47]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.formatting.rule import FormulaRule

# Load workbook
wb = load_workbook(file_path)
ws = wb.active

# Find the column index for 'signal'
for col in ws.iter_cols(1, ws.max_column):
    if col[0].value == "signal":
        signal_col_letter = col[0].column_letter
        break

# Define fills
red_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")   # light red
green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid") # light green

# Apply rules (starting from row 2 to skip header)
data_range = f"{signal_col_letter}2:{signal_col_letter}{ws.max_row}"

# SELL → Light Red
ws.conditional_formatting.add(
    data_range,
    FormulaRule(
        formula=[f'{signal_col_letter}2="SELL"'],
        fill=red_fill
    )
)

# BUY → Light Green
ws.conditional_formatting.add(
    data_range,
    FormulaRule(
        formula=[f'{signal_col_letter}2="BUY"'],
        fill=green_fill
    )
)

# Save
wb.save(file_path)